# Eff-3DPSeg (spconv) on Kaggle

Settings: **Accelerator = GPU** (T4/P100), **Internet = On**. Add your raw Pheno4D folder as a Kaggle Dataset (one sub-folder per plant, annotated files end in `_a.txt`).

Edit the two constants in the next cell.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/<your-username>/3d-phenomics.git'   # <- your fork
RAW_DIR  = '/kaggle/input/pheno4d'                                  # <- raw Pheno4D dataset

REPO = '/kaggle/working/3d-phenomics'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
%cd {REPO}/eff3dpseg

In [ ]:
# spconv publishes wheels only for some CUDA versions (PyPI: cu120, cu124, cu126), which can be
# older than the notebook's CUDA. Try the exact match first, then the newest older builds, and keep
# the first one that passes a real GPU test (run in a subprocess so a bad build can't poison this kernel).
import torch, subprocess, sys
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise SystemExit('No GPU / CPU-only torch. Notebook Settings -> Accelerator -> GPU (T4/P100), '
                     'then Run -> Restart session. spconv needs CUDA.')

GPU_TEST = '''
import torch, spconv.pytorch as spconv
idx = torch.tensor([[0,0,0,0],[0,1,0,0],[0,0,1,0],[0,1,1,0]], dtype=torch.int32).cuda()
x = spconv.SparseConvTensor(torch.ones(4, 1).cuda(), idx, [2, 2, 2], 1)
m = spconv.SubMConv3d(1, 2, 3, padding=1, indice_key="t").cuda()
assert m(x).features.shape == (4, 2)
print("spconv GPU test OK")
'''

def works():
    r = subprocess.run([sys.executable, '-c', GPU_TEST], capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr)[-1500:]

ok, log = works()
if ok:
    print('spconv already working')
else:
    exact = 'spconv-cu' + torch.version.cuda.replace('.', '')
    candidates = [exact] + [c for c in ('spconv-cu126', 'spconv-cu124') if c != exact]
    for pkg in candidates:
        print('trying', pkg)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'spconv', 'spconv-cu120',
                        'spconv-cu124', 'spconv-cu126', 'cumm-cu120', 'cumm-cu124', 'cumm-cu126'])
        if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg]).returncode != 0:
            print('  no such wheel for this Python/platform')
            continue
        ok, log = works()
        if not ok and int(torch.version.cuda.split('.')[0]) >= 13:
            # CUDA 12 builds need the CUDA 12 runtime libs, which a CUDA 13 torch does not ship.
            print('  retrying with CUDA 12 runtime libs')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                            'nvidia-cuda-runtime-cu12', 'nvidia-cuda-nvrtc-cu12'])
            ok, log = works()
        if ok:
            print('  works:', pkg)
            break
        print('  installed but the GPU test failed:\n', log)
    if not ok:
        raise SystemExit('No working spconv build here (CUDA %s, Python %s). Last error above.'
                         % (torch.version.cuda, sys.version_info[:2]))
print(log if ok else '')

In [ ]:
!python scripts/smoke_test.py

In [ ]:
# Convert raw txt -> npz + plant-level splits. Hold out whole plants for val/test.
# Plant names: ls $RAW_DIR   (edit --val-plants / --test-plants accordingly)
DATA = '/kaggle/working/pheno4d_npz'
!python scripts/prepare_pheno4d.py --raw {RAW_DIR} --out {DATA} --val-plants Tomato06 --test-plants Tomato07

In [ ]:
# --voxel-size is in the raw point-cloud units; the original used 0.2. Check your units!
!python scripts/train.py --data {DATA} --run-name demo --max-iter 2000 --validate-step 50 --time-limit-hours 11

### Resume after a session cut
Checkpoints live in `/kaggle/working/checkpoints` (save them as a Dataset between sessions). Re-run with:
`!python scripts/train.py --data {DATA} --run-name demo --max-iter 2000 --resume auto`

In [ ]:
!python scripts/infer.py --data {DATA} --ckpt /kaggle/working/checkpoints/demo_best.pth --phase test --out-dir /kaggle/working/preds --save-ply